# 📦 Publish Dataset → GitHub Release (`data-v8`)

**Jalankan di Colab.** Sekali jalan: clone repo (yang masih memuat dataset di git) →
generate `align_*.npy` → kemas tarball + MANIFEST → **upload sebagai aset Release `data-v8`**.
Setelah ini, notebook training menarik dataset dari Release (sel §2c), dan Anda bisa
menjalankan `maintenance_strip_dataset.ipynb` untuk membuang dataset dari git (repo ramping).

Prasyarat: PR v8 sudah di-merge ke `colab` (agar script `pack_dataset_release.py` /
`release_assets.py` / `make_align_variants.py` tersedia), dan `GITHUB_TOKEN` (PAT scope `repo`)
ada di Colab `userdata`.

Alur: setup → clone → deps → generate align → pack → upload → verifikasi round-trip.


## 1. Setup + Clone (branch `colab` — masih memuat dataset di git)


In [ ]:
from google.colab import userdata
from pathlib import Path
import subprocess, sys, os, time, glob

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_SLUG    = 'RZulfikri/Thesis'
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git'
COLAB_BRANCH = 'colab'
DATA_TAG     = 'data-v8'
REPO_DIR     = Path('/content/Thesis')
DATA_DIR     = REPO_DIR / '3DCNN' / 'dataset'
REL_DIR      = Path('/content/rel')          # output tarball (di luar repo)
REL_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    print('clone', COLAB_BRANCH, '...')
    subprocess.run(['git','clone','--single-branch','--branch',COLAB_BRANCH,REPO_URL,str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',COLAB_BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',COLAB_BRANCH], check=True)

sys.path.insert(0, str(REPO_DIR / '3DRegistration'))
_n_ply = len(glob.glob(str(DATA_DIR / '*/*/frame_*/output.ply')))
print(f'output.ply di clone: {_n_ply} frame')
assert _n_ply > 0, ('Dataset tidak ada di clone. Branch colab harus MASIH memuat dataset '
                    '(publish dilakukan SEBELUM strip-history).')



## 2. Dependencies (zstd untuk kompresi cepat + open3d untuk baca PLY)


In [ ]:
# zstd: kompresi tarball multi-thread (fallback gzip bila gagal).
subprocess.run(['apt-get','-qq','install','-y','zstd'], check=False)
try:
    import open3d  # noqa: F401
except ImportError:
    print('install open3d (untuk make_align_variants baca output.ply) ...')
    subprocess.run([sys.executable,'-m','pip','install','-q','open3d'], check=True)
print('zstd:', subprocess.run(['which','zstd'],capture_output=True,text=True).stdout.strip() or 'TIDAK ADA (pakai gzip)')



## 3. Generate `align_*.npy` (A1/A2/A4/A5)
Idempotent — frame yang sudah punya file align di-skip. Inilah satu-satunya tahap berat
yang menambah ~3.6 GB; setelah masuk Release, tak perlu generate lagi.


In [ ]:
t0=time.time()
r=subprocess.run([sys.executable,'make_align_variants.py','--data_dir',str(DATA_DIR)],
                 cwd=str(REPO_DIR / '3DRegistration'))
assert r.returncode==0, 'make_align_variants gagal'
for m in ['align_center','align_centerscale','align_pca_robust','align_anatomical']:
    n=len(glob.glob(str(DATA_DIR / f'*/*/frame_*/{m}.npy')))
    print(f'  {m}.npy: {n}/{_n_ply}')
print(f'align selesai ({time.time()-t0:.0f}s)')



## 4. Pack → tarball + MANIFEST
`pack_dataset_release.py` mengemas ply+geometry+cnn_input+fps+align → `dataset_v8.tar.zst`
(otomatis di-split bila > 1.9 GB) + `MANIFEST_v8.json` (sha256 tiap part).


In [ ]:
t0=time.time()
r=subprocess.run([sys.executable,'pack_dataset_release.py',
                  '--data_dir',str(DATA_DIR),'--version','v8','--out_dir',str(REL_DIR)],
                 cwd=str(REPO_DIR / '3DRegistration'))
assert r.returncode==0, 'pack gagal'
produced=sorted(str(p) for p in REL_DIR.iterdir())
print(f'pack selesai ({time.time()-t0:.0f}s). File:')
for p in produced: print('  ', p, f'{Path(p).stat().st_size/1e6:.1f}MB')



## 5. Upload sebagai Release `data-v8`
Meng-upload SEMUA part + MANIFEST. Bila tag sudah ada, aset senama ditimpa.
(Untuk versi data baru nanti: ubah `--version` & `DATA_TAG` jadi v9 dst, lalu hapus v8.)


In [ ]:
import importlib, release_assets as ra; importlib.reload(ra)
files=sorted(str(p) for p in REL_DIR.iterdir())
ra.create_or_get_release(REPO_SLUG, DATA_TAG, GITHUB_TOKEN,
                         name='Dataset v8 (point cloud + align variants)',
                         body='Tarball dataset 3D palm (output.ply + geometry.json + '
                              'cnn_input*.npy + align_*.npy) + MANIFEST sha256. '
                              'Ditarik otomatis oleh sel Data Bootstrap notebook v8.')
ra.upload_assets(REPO_SLUG, DATA_TAG, files, GITHUB_TOKEN)
print('\n✅ Upload selesai. Cek tab Releases:', f'https://github.com/{REPO_SLUG}/releases/tag/{DATA_TAG}')



## 6. Verifikasi round-trip (unduh ke dir bersih)
Membuktikan Release bisa ditarik & utuh (sha256) — aman sebelum strip-history.


In [ ]:
VER = Path('/content/verify_pull')
import shutil
if VER.exists(): shutil.rmtree(VER)
manifest = ra.pull_dataset(REPO_SLUG, DATA_TAG, VER, GITHUB_TOKEN, workdir='/content/_verify_dl')
got = len(glob.glob(str(VER / '*/*/frame_*/*')))
print(f'\nround-trip: {got} file diextract (manifest {manifest["file_count"]}).')
assert got >= manifest['file_count'], 'jumlah file extract < manifest — cek!'
print('✅ Release data-v8 valid & lengkap.')
print('Langkah berikut (opsional): jalankan maintenance_strip_dataset.ipynb untuk membuang')
print('dataset dari git history → repo ramping. Dataset tetap aman di Release ini.')

